In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import statsmodels.api as sm

# Confidence and Prediction Intervals

## Generate Data
Start with toy data
$$
y_i = 2 - 1.5 x_i + \text{noise}
$$

In [ ]:
#experiment with different numbers of points
# n = 15
n = 1000


x_ = np.linspace(0, 10, n)
rng = np.random.default_rng(1234)
y_ = 2 - 1.5 * x_ + rng.normal(size=x_.shape)

Store in a data frame for later:

In [ ]:
df = pd.DataFrame({'x': x_, 'y': y_})
df.head()

## Fit with statsmodels

In [ ]:
# build design matrix
X = sm.add_constant(df['x'])  # Adds a constant term to the predictor

# OLS = Ordinary Least Squares, specify the response variable and the design matrix
model = sm.OLS(df['y'], X) 

# Fit the model
results = model.fit()

results.summary()

### Confidence Intervals

In [ ]:
preds = results.get_prediction(X) # this is the predicted values at the training points.  This could be a different matrix X of values
preds

In [ ]:
X

In [ ]:
preds.predicted_mean

In [ ]:
ci_vals = preds.conf_int() # defaults to the 95% confidence interval.
# ci_vals = preds.conf_int(alpha=0.01) # for 99% confidence intervals
ci_vals

In [ ]:
fig, ax = plt.subplots()
df.plot.scatter(x='x', y='y', ax=ax, label='Data')
ax.plot(df['x'], preds.predicted_mean, label='OLS Prediction', color='C1')
ax.fill_between(
    df['x'],
    ci_vals[:,0], # lower bound
    ci_vals[:,1], # upper bound
    color='C1',
    alpha=0.25,     # transparency
    label='95% CI',
)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.set_title('Confidence Intervals for OLS')
ax.legend()


### Prediction Intervals

In [ ]:
pi_vals = preds.conf_int(obs=True) # defaults to the 95% confidence interval.
# pi_vals = preds.conf_int(obs=True, alpha=0.01) # for 99% confidence intervals
pi_vals

In [ ]:
fig, ax = plt.subplots()
df.plot.scatter(x='x', y='y', ax=ax, label='Data')
ax.plot(df['x'], preds.predicted_mean, label='OLS Prediction', color='C1')
ax.fill_between(
    df['x'],
    ci_vals[:,0], # lower bound
    ci_vals[:,1], # upper bound
    color='C1',
    alpha=0.25,     # transparency
    label='95% CI',
)

ax.fill_between(
    df['x'],
    pi_vals[:,0], # lower bound
    pi_vals[:,1], # upper bound
    color='C2',
    alpha=0.25,     # transparency
    label='95% PI',
)


ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.set_title('Confidence Intervals for OLS')
ax.legend()


* Confidence intervals reflect uncertainty in the mean trend line
* Prediction intervals reflect uncertainty in what would be expected in a new observation (a sample, not the mean)

# Fitting with Named Features

In [ ]:
import statsmodels.formula.api as smf

This lets us build the models directly using the named features, **without** first constructing a design matrix:

In [ ]:
df.head()

In [ ]:
model = smf.ols('y ~ x', df)
model

This implicitly sets up for fitting the model
$$
y \approx \beta_0 + \beta_1 x
$$

In [ ]:
results = model.fit()
results.summary()

If you want the design matrix out:

In [ ]:
results.model.exog

NO need to pad data to make new predictions:

In [ ]:
new_x = pd.DataFrame({'x': [2.5, 5.5, 8.5]})
predictions = results.predict(new_x)
predictions